# Capstone Visualization — All Charts
Resource Curse & Economic Complexity | Moody's Capstone Project

Charts are saved to `Final/charts/` as interactive HTML files and `Final/charts_png/` as static PNGs.
Run `scripts/export_charts_png.py` after this notebook to regenerate PNGs.

In [ ]:
import os, sys, warnings
#os.chdir('/Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP')
os.chdir('/Users/Usuario/Github/Capstone/FINAL CODE RECAP')
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import r2_score, mean_squared_error

from viz_utils import (
    CLUSTER_LABELS, CLUSTER_COLORS, INCLUDE_LIST,
    WRITE_CONFIG, BG, GRID, FONT, NAVY, PALETTE,
    load_master, load_master_wide, load_clusters, load_nr,
    shorten_feat, base_layout, save,
    analyze_country_missingness,
)

OUT = 'Final/charts'
NB5 = 'Final/NB5'
os.makedirs(OUT, exist_ok=True)

## Section 1 — Clustering (NB4)

In [ ]:
# # Clustering: Natural Resource Profiles
# 
# **Capstone Project — Moody's Ratings**  
# *Pipeline step 4 of 5: Identifying resource-dependency typologies*
# 
# This notebook classifies countries into four natural resource profile groups using PCA and K-Means clustering on per-capita production values. The clustering is run for three time windows to capture both cross-sectional structure and temporal shifts.
# 
# **Methodology:**
# 1. Production values (quantity x price, in USD) are divided by population to obtain per-capita figures
# 2. A log(1+x) transformation compresses the extreme right skew typical of resource data
# 3. PCA reduces the feature space to 2 components (PC1 captures hydrocarbons, PC2 captures minerals)
# 4. K-Means (k=4) partitions countries in PCA space into four groups
# 
# **Why these choices:**
# - *Per capita* (not per GDP): avoids mixing resource profiles with economic structure, which is what we're trying to predict (ECI)
# - *log1p* (not z-scoring): appropriate for distributions with 1000:1 ratios between smallest and largest producers; StandardScaler would compress most countries into a narrow band
# - *PCA then K-Means* (not K-Means on raw features): removes correlated noise (oil and gas co-occur), reduces dimensionality for better Euclidean distance behaviour
# - *k=4*: validated with silhouette analysis; produces a clean typology matching the economic literature
# 
# **Three variants:**
# - **1995 snapshot:** Cluster assignments based on 1995 production profiles (baseline year)
# - **2019 snapshot:** Cluster assignments based on 2019 production profiles (end of panel)
# - **Aggregated:** Uses earliest available year per country across 1995-2005
# 
# **Inputs:**
# - `intermediary/NaturalResource.csv` (from Step 3)
# - `intermediary/Master.csv` (from Step 3, for the ECI evolution chart)
# 
# **Outputs:**
# - `intermediary/clusters1995.csv`, `intermediary/clusters2019.csv`, `intermediary/clustersagg.csv`
# 
# ---

# ## 0. Setup

# Fixed colour palette keyed by cluster label (label-stable, not cluster-ID-stable)
_LABEL_COLORS = {
    'Petrostates':          '#d4853b',   # orange
    'Oil Exporters':        '#4a6fa5',   # blue
    'Diversified Exporters':'#2e7d4a',   # green
    'Gold & Coal':          '#c23a3a',   # red
}

# ## 1. Load Data and Define Sample
# 
# The 54-country sample corresponds to resource-dependent developing economies identified through the filtering criteria in Steps 1-3 (total natural resource rents > 5% of GDP in 1995, non-high-income).

nr = pd.read_csv("intermediary/NaturalResource.csv")

# Countries in the analysis sample
include_list = [
    'AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
    'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
    'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
    'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
    'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
    'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE',
]

nr_sample = nr[nr["Country Code"].isin(include_list)]
print(f"NR data: {nr_sample.shape[0]:,} rows")
print(f"Countries: {nr_sample['Country Code'].nunique()}")
print(f"Years: {sorted(nr_sample['Year'].unique())}")
print(f"Resources: {nr_sample['Resource'].nunique()}")

# ## 2. Clustering Pipeline
# 
# The clustering function encapsulates the full pipeline: pivot to per-capita production values, log1p transform, PCA, K-Means. It returns the cluster assignments along with PC coordinates for plotting.
# 
# Cluster labels are assigned automatically based on the centroid position in PCA space rather than hardcoded to specific cluster numbers (since K-Means assigns arbitrary IDs that can change between runs). The labelling logic uses the sign and magnitude of PC1 and PC2 loadings: PC1 captures hydrocarbon dominance, PC2 captures mineral dominance.

def run_clustering(nr_data, year_filter=None, agg_years=None, n_clusters=4, random_state=42):
    """
    Full clustering pipeline: pivot -> per capita -> log1p -> PCA(2) -> KMeans(k).

    Parameters
    ----------
    nr_data : DataFrame
        NaturalResource data with Country, Country Code, Year, Resource,
        Production_TotalValue, Population columns.
    year_filter : int or None
        If set, restrict to a single year (e.g. 1995 or 2019).
    agg_years : list or None
        If set, restrict to these years and take earliest per country.
    n_clusters : int
        Number of K-Means clusters.
    random_state : int
        For KMeans reproducibility (PCA uses deterministic solver).

    Returns
    -------
    pca_df : DataFrame with Country, Country Code, Year, PC1, PC2, Cluster, ClusterLabels
    pca_model : fitted PCA object
    feature_cols : list of resource column names used
    """

    df = nr_data.copy()

    # ── Year selection ──
    if year_filter is not None:
        df = df[df["Year"] == year_filter]
    elif agg_years is not None:
        df = df[df["Year"].isin(agg_years)]

    # ── Pivot: one row per country, one column per resource ──
    df_pivot = df.pivot_table(
        index=["Country", "Country Code", "Year", "Population"],
        columns="Resource",
        values="Production_TotalValue",
    ).reset_index()

    resource_cols = df_pivot.columns.difference(
        ["Country", "Country Code", "Year", "Population"]
    )

    # ── Per-capita normalisation ──
    df_pivot[resource_cols] = df_pivot[resource_cols].div(
        df_pivot["Population"], axis=0
    )
    df_pivot.drop(columns="Population", inplace=True)
    df_pivot = df_pivot.fillna(0)

    # ── Take earliest year per country (for aggregated variant) ──
    df_latest = (
        df_pivot.sort_values("Year", ascending=True)
        .groupby(["Country", "Country Code"])
        .first()
        .reset_index()
    )

    feature_cols = [c for c in df_latest.columns if c not in ["Country", "Country Code", "Year"]]

    # ── log1p transform ──
    X = df_latest[feature_cols].fillna(0)
    X_log = np.log1p(X)

    # ── PCA ──
    pca = PCA(n_components=2)
    pca_components = pca.fit_transform(X_log)

    # ── K-Means ──
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    clusters = kmeans.fit_predict(pca_components)

    # ── Build results DataFrame ──
    pca_df = pd.DataFrame({
        "Country": df_latest["Country"],
        "Country Code": df_latest["Country Code"],
        "Year": df_latest["Year"],
        "PC1": pca_components[:, 0],
        "PC2": pca_components[:, 1],
        "Cluster": clusters,
    })

    # ── Auto-label clusters from centroids ──
    # FIX (Check 11): replaced hardcoded PC1/PC2 thresholds with rank-based labeling.
    # The old thresholds (pc1 > 1.5, etc.) were calibrated on data where oil production
    # was not annualised; after the unit fix in NB3 the scale changes and hardcoded
    # absolute values no longer generalise. Rank-based assignment is invariant to scale.
    #
    # Logic:
    #   1. Cluster with highest PC1 → Oil-dominant (hydrocarbons load on PC1)
    #   2. Cluster with highest PC2 (not already labeled) → Mineral-dominant
    #   3. Remaining cluster with higher PC1 → Some Oil
    #   4. Remaining cluster with lower PC1 → No Resources
    centroids = kmeans.cluster_centers_

    pc1_rank = list(np.argsort(-centroids[:, 0]))  # highest PC1 first
    pc2_rank = list(np.argsort(-centroids[:, 1]))  # highest PC2 first

    label_map = {}
    labeled = set()

    # 1. Highest PC1 → Oil-dominant (Gulf-style petrostates, high per-capita oil)
    oil_id = pc1_rank[0]
    label_map[oil_id] = "Petrostates"
    labeled.add(oil_id)

    # 2. Highest PC2 not yet labeled → Mineral/diversified (copper, coal, gold mix)
    mineral_id = next(c for c in pc2_rank if c not in labeled)
    label_map[mineral_id] = "Diversified Exporters"
    labeled.add(mineral_id)

    # 3 & 4. Remaining two by PC1 rank
    remaining = [c for c in pc1_rank if c not in labeled]
    label_map[remaining[0]] = "Oil Exporters"
    label_map[remaining[1]] = "Gold & Coal"

    pca_df["ClusterLabels"] = pca_df["Cluster"].map(label_map)

    # ── Metrics ──
    sil = silhouette_score(pca_components, clusters)
    print(f"Silhouette score: {sil:.3f}")
    print(f"Cluster distribution:")
    for cid in sorted(pca_df["Cluster"].unique()):
        n = (pca_df["Cluster"] == cid).sum()
        print(f"  {label_map[cid]}: {n} countries")

    return pca_df, pca, feature_cols

# ## 3. Validate k with Silhouette Analysis
# 
# Before running the final clustering, we check that k=4 is a reasonable choice by computing silhouette scores for k=2 through k=8. The silhouette score measures how similar each point is to its own cluster compared to neighbouring clusters (range: -1 to 1, higher is better).

# Run validation on 1995 data
nr_1995 = nr_sample[nr_sample["Year"] == 1995].copy()

df_pivot_val = nr_1995.pivot_table(
    index=["Country", "Country Code", "Year", "Population"],
    columns="Resource",
    values="Production_TotalValue",
).reset_index()

resource_cols_val = df_pivot_val.columns.difference(
    ["Country", "Country Code", "Year", "Population"]
)
df_pivot_val[resource_cols_val] = df_pivot_val[resource_cols_val].div(
    df_pivot_val["Population"], axis=0
)
df_pivot_val = df_pivot_val.fillna(0)

feat_cols_val = [c for c in df_pivot_val.columns
                 if c not in ["Country", "Country Code", "Year", "Population"]]
X_val = np.log1p(df_pivot_val[feat_cols_val].fillna(0))

pca_val = PCA(n_components=2)
X_pca_val = pca_val.fit_transform(X_val)

# Silhouette scores for k = 2..8
k_range = range(2, 9)
sil_scores = []
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca_val)
    sil_scores.append(silhouette_score(X_pca_val, labels))
    inertias.append(km.inertia_)

print("\nSilhouette scores:")
for k, s in zip(k_range, sil_scores):
    marker = " <-- selected" if k == 4 else ""
    print(f"  k={k}: {s:.3f}{marker}")

# ## 4. Run Clustering for All Three Variants
# 
# The pipeline is run three times:
# - **1995:** Single-year snapshot at the start of the panel
# - **2019:** Single-year snapshot at the end of the panel
# - **Aggregated:** Uses data from 1995, 1999, and 2005, taking the earliest available year per country. This smooths out year-specific fluctuations while still capturing structural resource profiles

results = {}

# ── 1995 snapshot ──
print("=" * 60)
print("1995 SNAPSHOT")
print("=" * 60)
pca_1995, pca_model_1995, feat_1995 = run_clustering(
    nr_sample, year_filter=1995
)
results["1995"] = pca_1995
print()

# ── 2019 snapshot ──
print("=" * 60)
print("2019 SNAPSHOT")
print("=" * 60)
pca_2019, pca_model_2019, feat_2019 = run_clustering(
    nr_sample, year_filter=2019
)
results["2019"] = pca_2019
print()

# ── Aggregated (1995, 1999, 2005) ──
print("=" * 60)
print("AGGREGATED (1995, 1999, 2005)")
print("=" * 60)
pca_agg, pca_model_agg, feat_agg = run_clustering(
    nr_sample, agg_years=[1995, 1999, 2005]
)
results["agg"] = pca_agg

# ## 5. PCA Loadings Analysis
# 
# The loadings reveal which resources drive each principal component. PC1 is expected to capture hydrocarbon abundance (oil, natural gas), while PC2 should capture mineral production (copper, gold, zinc, etc.). This interpretation is central to the cluster labelling logic.

# Use 1995 model for loadings analysis (consistent with report)
loadings = pd.DataFrame(
    pca_model_1995.components_.T,
    columns=["PC1", "PC2"],
    index=feat_1995,
)

print("PCA Explained Variance (1995):")
for i, var in enumerate(pca_model_1995.explained_variance_ratio_):
    cum = pca_model_1995.explained_variance_ratio_[:i + 1].sum()
    print(f"  PC{i+1}: {var*100:.1f}% (cumulative: {cum*100:.1f}%)")

print("\nTop loadings by component:")
for pc in ["PC1", "PC2"]:
    print(f"\n{pc}:")
    sorted_l = loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index)
    for feat, val in sorted_l.head(8).items():
        print(f"  {feat:35s} {val:+.4f}")

# ## 6. Biplot: PCA Space with Cluster Assignments
# 
# The biplot overlays cluster assignments onto the PCA space, with arrows showing the direction and strength of each resource's contribution to the principal components. This is Figure 2 in the report.

def create_biplot(pca_df, pca_model, feature_cols, title_suffix=""):
    """Create PCA biplot with cluster colours and loading arrows."""

    loadings_plot = pca_model.components_.T * np.sqrt(pca_model.explained_variance_)
    loadings_df = pd.DataFrame(loadings_plot[:, :2], columns=["PC1", "PC2"], index=feature_cols)
    scale_factor = 2.5
    loadings_scaled = loadings_df * scale_factor

    # Top features by combined loading magnitude
    importance = loadings_df.abs().sum(axis=1)
    top_n = min(15, len(feature_cols))
    top_feats = importance.nlargest(top_n).index

    fig = px.scatter(
        pca_df, x="PC1", y="PC2",
        color="ClusterLabels",
        hover_data=["Country", "Country Code", "Year"],
        color_discrete_sequence=px.colors.qualitative.Bold,
    )

    for feat in top_feats:
        fig.add_annotation(
            x=loadings_scaled.loc[feat, "PC1"],
            y=loadings_scaled.loc[feat, "PC2"],
            ax=0, ay=0, xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="black",
        )
        fig.add_annotation(
            x=loadings_scaled.loc[feat, "PC1"] * 1.15,
            y=loadings_scaled.loc[feat, "PC2"] * 1.15,
            text=feat, showarrow=False,
            font=dict(size=9, color="black"),
        )

    var1 = pca_model.explained_variance_ratio_[0] * 100
    var2 = pca_model.explained_variance_ratio_[1] * 100
    fig.update_layout(
        width=1000, height=700,
        xaxis_title=f"PC1 ({var1:.1f}%)",
        yaxis_title=f"PC2 ({var2:.1f}%)",
    )
    return fig


fig_biplot = create_biplot(pca_1995, pca_model_1995, feat_1995, " — 1995")
fig_biplot.write_html(os.path.join(OUT, '03_cluster__pca_biplot_country_resource_groups.html'))
print(f'Saved: Final/charts/03_cluster__pca_biplot_country_resource_groups.html')

# ## 7. Choropleth Map with Dominance Flags
# 
# Countries producing more than 15% of global output for any single resource are flagged with a red border. This highlights major producers (e.g. Chile for copper, Saudi Arabia for oil) whose resource profiles have global significance.

def create_cluster_map(pca_df, nr_data, cluster_names_map=None, dominance_threshold=15.0):
    """Choropleth map with red borders for major global producers."""

    if cluster_names_map is None:
        cluster_names_map = dict(
            zip(pca_df["Cluster"].unique(), pca_df["ClusterLabels"].unique())
        )

    # ── Global production shares ──
    df_total = nr_data.pivot_table(
        index=["Country", "Country Code"],
        columns="Resource",
        values="Production_TotalValue",
        aggfunc="sum",
    ).reset_index().fillna(0)

    prod_cols = [c for c in df_total.columns if c not in ["Country", "Country Code"]]
    for col in prod_cols:
        total = df_total[col].sum()
        if total > 0:
            df_total[f"{col}_Share"] = (df_total[col] / total) * 100

    share_cols = [c for c in df_total.columns if c.endswith("_Share")]

    # Merge shares into pca_df
    df_map = pca_df.merge(df_total[["Country Code"] + share_cols], on="Country Code", how="left")

    # Flag dominant producers (vectorised)
    df_map["Is_Dominant"] = (df_map[share_cols] >= dominance_threshold).any(axis=1)
    df_map["Dominant_Resources"] = df_map.apply(
        lambda row: [
            sc.replace("_Share", "")
            for sc in share_cols
            if row.get(sc, 0) >= dominance_threshold
        ],
        axis=1,
    )

    # ── Hover text ──
    def make_hover(row):
        lbl = row["ClusterLabels"]
        lines = [f"<b>{row['Country']}</b>", f"Cluster: {lbl}"]
        # Top resources
        vals = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        if vals:
            lines.append("<br>Top Resources:")
            for res, v in vals[:3]:
                if v > 1e9:
                    lines.append(f"  {res}: ${v/1e9:.1f}B")
                elif v > 1e6:
                    lines.append(f"  {res}: ${v/1e6:.0f}M")
                else:
                    lines.append(f"  {res}: ${v:,.0f}")
        return "<br>".join(lines)

    df_map["hover_text"] = df_map.apply(make_hover, axis=1)

    # ── Build map ──
    fig = go.Figure()

    for cid in sorted(df_map["Cluster"].unique()):
        lbl   = cluster_names_map.get(cid, f"Cluster {cid}")
        color = _LABEL_COLORS.get(lbl, '#aaa')

        # Non-dominant countries
        sub = df_map[(df_map["Cluster"] == cid) & (~df_map["Is_Dominant"])]
        if len(sub) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub["Country Code"], z=[cid]*len(sub),
                colorscale=[[0, color], [1, color]], showscale=False,
                showlegend=True,
                customdata=sub["hover_text"].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=lbl,
                marker=dict(line=dict(color="white", width=0.6)),
            ))

        # Dominant producers — thick dark border
        sub_d = df_map[(df_map["Cluster"] == cid) & (df_map["Is_Dominant"])]
        if len(sub_d) > 0:
            fig.add_trace(go.Choropleth(
                locations=sub_d["Country Code"], z=[cid]*len(sub_d),
                colorscale=[[0, color], [1, color]], showscale=False,
                showlegend=False,           # same cluster — don't duplicate legend entry
                customdata=sub_d["hover_text"].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{lbl} ★ major producer",
                marker=dict(line=dict(color="#111", width=2.2)),
            ))

    # Dummy legend entry explaining the black border
    fig.add_trace(go.Choropleth(
        locations=["ZZZ"], z=[0],
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
        showscale=False, showlegend=True,
        name="★ >15% of global output",
        marker=dict(line=dict(color="#111", width=2.2)),
    ))

    fig.update_geos(
        projection_type="natural earth",
        showcountries=True, countrycolor="#ccc",
        showcoastlines=True, coastlinecolor="#ccc",
        showland=True, landcolor="#ffffff",
        showocean=True, oceancolor="#ffffff",
        showframe=False,
    )
    fig.update_layout(
        width=1200, height=520,
        margin=dict(l=0, r=0, t=50, b=70),
        legend=dict(
            orientation="h",
            x=0.5, y=-0.08, xanchor="center", yanchor="top",
            font=dict(size=11, family="IBM Plex Sans"),
            bgcolor="rgba(250,250,250,0.9)",
            bordercolor="#ffffff", borderwidth=1,
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family="IBM Plex Sans"),
    )
    return fig


# Use 1995 for the main map (matches report Figure 3)
nr_1995_full = nr_sample[nr_sample["Year"] == 1995]
fig_map = create_cluster_map(pca_1995, nr_1995_full)
fig_map.write_html(os.path.join(OUT, '04_cluster__world_map_four_resource_profiles.html'))
print(f'Saved: Final/charts/04_cluster__world_map_four_resource_profiles.html')

# ## 8. ECI vs GDP Evolution (Rosling Chart)
# 
# This animated chart tracks how each country's Economic Complexity Index and GDP per capita evolved from 1995 to 2019. Countries are coloured by their 1995 cluster assignment (fixed throughout the animation), with bubble size proportional to production per capita. Arrows trace each country's trajectory from its 1995 starting position. This is Figure 4 in the report.

master = pd.read_csv("intermediary/Master.csv")
master = master[master["Country Code"].isin(include_list)]

# Merge cluster assignments (from aggregated clustering)
master = pd.merge(
    master,
    pca_agg[["Country Code", "Cluster", "ClusterLabels"]],
    on="Country Code",
    how="left",
)

CLUSTER_COLORS_NB4 = {
    cid: _LABEL_COLORS.get(
        pca_agg.loc[pca_agg["Cluster"] == cid, "ClusterLabels"].iloc[0], '#aaa'
    )
    for cid in sorted(pca_agg["Cluster"].unique())
}

CLUSTER_NAMES = dict(zip(pca_agg["Cluster"], pca_agg["ClusterLabels"]))

def create_rosling_chart(df, cluster_colors, cluster_names, arrow_opacity=0.5, arrow_width=2):
    """Animated ECI vs log(GDP pc) chart with trajectory arrows from 1995."""

    data = df.copy()
    data["Log GDP per capita"] = np.log(data["GDP per capita (constant prices, PPP)"])
    data["Production_Per_Capita"] = data["Total_Production_Value"] / data["Population"]

    # Fix cluster to 1995 value
    c1995 = data[data["Year"] == 1995][["Country Code", "Cluster"]].copy()
    c1995 = c1995.rename(columns={"Cluster": "Cluster_1995"})
    data = data.merge(c1995, on="Country Code", how="left")
    data = data.dropna(subset=["Cluster_1995", "Log GDP per capita",
                                "Economic Complexity Index", "Production_Per_Capita"])
    data["Cluster_1995"] = data["Cluster_1995"].astype(int)

    # Bubble size
    data["Bubble_Size"] = np.sqrt(data["Production_Per_Capita"])
    mn, mx = data["Bubble_Size"].min(), data["Bubble_Size"].max()
    data["Bubble_Size_Scaled"] = 8 + (data["Bubble_Size"] - mn) / (mx - mn) * 42

    data = data.sort_values(["Year", "Country Code"])
    years = sorted(data["Year"].unique())
    countries_list = data["Country Code"].unique()
    clusters = sorted(data["Cluster_1995"].unique())

    # Build per-country data
    cdata = {}
    for code in countries_list:
        cdf = data[data["Country Code"] == code].sort_values("Year")
        origin = cdf[cdf["Year"] == 1995]
        if len(origin) == 0:
            continue
        cdata[code] = {
            "years": cdf["Year"].values,
            "x": cdf["Log GDP per capita"].values,
            "y": cdf["Economic Complexity Index"].values,
            "x0": origin["Log GDP per capita"].values[0],
            "y0": origin["Economic Complexity Index"].values[0],
            "size": cdf["Bubble_Size_Scaled"].values,
            "name": cdf["Country Name"].iloc[0],
            "cluster": cdf["Cluster_1995"].iloc[0],
            "prod_pc": cdf["Production_Per_Capita"].values,
        }

    valid_countries = list(cdata.keys())
    first_year = years[0]

    fig = go.Figure()

    for cl in clusters:
        cc = [c for c in valid_countries if cdata[c]["cluster"] == cl]
        color = cluster_colors.get(cl, "#999999")

        for code in cc:
            cd = cdata[code]
            idx = np.where(cd["years"] == first_year)[0]
            xc = cd["x"][idx[0]] if len(idx) > 0 else cd["x0"]
            yc = cd["y"][idx[0]] if len(idx) > 0 else cd["y0"]
            fig.add_trace(go.Scatter(
                x=[cd["x0"], xc], y=[cd["y0"], yc],
                mode="lines", line=dict(color=color, width=arrow_width),
                opacity=arrow_opacity, legendgroup=f"cl_{cl}", showlegend=False, hoverinfo="skip",
            ))

        for code in cc:
            cd = cdata[code]
            idx = np.where(cd["years"] == first_year)[0]
            if len(idx) > 0:
                i = idx[0]
                xv, yv, sv, pv = [cd["x"][i]], [cd["y"][i]], cd["size"][i], cd["prod_pc"][i]
            else:
                xv, yv, sv, pv = [cd["x0"]], [cd["y0"]], 15, 0
            fig.add_trace(go.Scatter(
                x=xv, y=yv, mode="markers+text",
                marker=dict(size=sv, color=color, opacity=0.85, line=dict(width=1.5, color="white")),
                text=[code], textposition="top center", textfont=dict(size=8, color="black"),
                name=cluster_names.get(cl, f"Cluster {cl}"),
                legendgroup=f"cl_{cl}", showlegend=(code == cc[0]),
                customdata=[[cd["name"], pv, first_year]],
                hovertemplate="<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>"
                              "ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>"
                              "Year: %{customdata[2]}<extra></extra>",
            ))

        for code in cc:
            cd = cdata[code]
            fig.add_trace(go.Scatter(
                x=[cd["x0"]], y=[cd["y0"]], mode="markers",
                marker=dict(size=5, color=color, opacity=0.6, symbol="circle"),
                legendgroup=f"cl_{cl}", showlegend=False, hoverinfo="skip",
            ))

    # Frames for animation
    frames = []
    for year in years:
        fd = []
        for cl in clusters:
            cc = [c for c in valid_countries if cdata[c]["cluster"] == cl]
            color = cluster_colors.get(cl, "#999999")
            for code in cc:
                cd = cdata[code]
                idx = np.where(cd["years"] == year)[0]
                if len(idx) > 0:
                    xc, yc = cd["x"][idx[0]], cd["y"][idx[0]]
                else:
                    mask = cd["years"] <= year
                    li = np.where(mask)[0][-1] if mask.any() else 0
                    xc, yc = cd["x"][li], cd["y"][li]
                fd.append(go.Scatter(x=[cd["x0"], xc], y=[cd["y0"], yc],
                                     mode="lines", line=dict(color=color, width=arrow_width), opacity=arrow_opacity))
            for code in cc:
                cd = cdata[code]
                idx = np.where(cd["years"] == year)[0]
                if len(idx) > 0:
                    i = idx[0]
                    xv, yv, sv, pv = [cd["x"][i]], [cd["y"][i]], cd["size"][i], cd["prod_pc"][i]
                else:
                    mask = cd["years"] <= year
                    if mask.any():
                        li = np.where(mask)[0][-1]
                        xv, yv, sv, pv = [cd["x"][li]], [cd["y"][li]], cd["size"][li], cd["prod_pc"][li]
                    else:
                        xv, yv, sv, pv = [cd["x0"]], [cd["y0"]], 15, 0
                fd.append(go.Scatter(
                    x=xv, y=yv, mode="markers+text",
                    marker=dict(size=sv, color=color, opacity=0.85, line=dict(width=1.5, color="white")),
                    text=[code], textposition="top center", textfont=dict(size=8),
                    customdata=[[cd["name"], pv, year]],
                    hovertemplate="<b>%{customdata[0]}</b><br>Log GDP pc: %{x:.2f}<br>"
                                  "ECI: %{y:.2f}<br>Prod/capita: $%{customdata[1]:,.0f}<br>"
                                  "Year: %{customdata[2]}<extra></extra>",
                ))
            for code in cc:
                cd = cdata[code]
                fd.append(go.Scatter(x=[cd["x0"]], y=[cd["y0"]], mode="markers",
                                     marker=dict(size=5, color=color, opacity=0.6, symbol="circle")))
        frames.append(go.Frame(data=fd, name=str(year)))

    fig.frames = frames

    eci_vals = data["Economic Complexity Index"]
    x_vals = data["Log GDP per capita"]
    fig.update_layout(
        xaxis=dict(range=[x_vals.min()-0.2, x_vals.max()+0.2], title="Log GDP per capita (PPP)"),
        yaxis=dict(range=[eci_vals.min()-0.5, eci_vals.max()+0.5], title="Economic Complexity Index"),
        plot_bgcolor="white", width=850, height=650,
        legend=dict(title="Resource Profile (1995)", x=1.02, y=0.99),
        updatemenus=[dict(
            type="buttons", showactive=True, x=1.0, y=-0.02,
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, dict(frame=dict(duration=500, redraw=True), transition=dict(duration=300))]),
                dict(label="Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            active=0, len=0.85, x=0.05, y=-0.12,
            currentvalue=dict(prefix="Year: ", font=dict(size=14)),
            steps=[dict(args=[[str(y)], dict(frame=dict(duration=300, redraw=True), mode="immediate")],
                        method="animate", label=str(y)) for y in years],
        )],
    )
    return fig


fig_rosling = create_rosling_chart(master, CLUSTER_COLORS_NB4, CLUSTER_NAMES)
fig_rosling.write_html(os.path.join(OUT, '05_cluster__eci_vs_gdp_animated_1995_to_2019.html'))
print(f'Saved: Final/charts/05_cluster__eci_vs_gdp_animated_1995_to_2019.html')

# ## 9. Add Hover Text and Export
# 
# The cluster CSVs include hover text with top resources for each country, matching the format used in the original analysis files.

def add_hover_text(pca_df, nr_data):
    """Add hover text with top resources to cluster DataFrame."""
    # Get total production by country and resource
    totals = nr_data.pivot_table(
        index=["Country", "Country Code"],
        columns="Resource",
        values="Production_TotalValue",
        aggfunc="sum",
    ).reset_index().fillna(0)

    prod_cols = [c for c in totals.columns if c not in ["Country", "Country Code"]]

    df = pca_df.merge(totals, on="Country Code", how="left", suffixes=("", "_nr"))

    def make_ht(row):
        lines = [f"<b>{row['Country']}</b>",
                 f"Cluster: {row['Cluster']}"]
        lines.append("<br>Top Resources:")
        vals = [(c, row.get(c, 0)) for c in prod_cols if row.get(c, 0) > 0]
        vals.sort(key=lambda x: x[1], reverse=True)
        for res, v in vals[:3]:
            lines.append(f"  {res}: ${v:,.0f}")
        return "<br>".join(lines)

    df["hover_text"] = df.apply(make_ht, axis=1)
    return df[["Country", "Country Code", "Year", "PC1", "PC2",
               "Cluster", "ClusterLabels", "hover_text"]]


# Export each variant
for label, df, nr_subset in [
    ("1995", results["1995"], nr_sample[nr_sample["Year"] == 1995]),
    ("2019", results["2019"], nr_sample[nr_sample["Year"] == 2019]),
    ("agg",  results["agg"],  nr_sample[nr_sample["Year"].isin([1995, 1999, 2005])]),
]:
    out = add_hover_text(df, nr_subset)
    out_path = f"intermediary/clusters{label}.csv"
    out.to_csv(out_path, index=False)
    print(f"Saved {out_path}: {len(out)} countries")

print("\nDone. All cluster CSVs exported.")

# ── NB4 SUMMARY ──
print("=" * 70)
print("NB4: CLUSTERING SUMMARY")
print("=" * 70)

print(f"\nPCA variance explained (1995): PC1={pca_model_1995.explained_variance_ratio_[0]*100:.1f}%, PC2={pca_model_1995.explained_variance_ratio_[1]*100:.1f}%")

for label, df in results.items():
    print(f"\n--- {label} ({len(df)} countries) ---")
    for cid in sorted(df['Cluster'].unique()):
        sub = df[df['Cluster'] == cid]
        lbl = sub['ClusterLabels'].iloc[0]
        codes = ', '.join(sorted(sub['Country Code'].tolist()))
        print(f"  {lbl} (n={len(sub)}): {codes}")

print(f"\nSaved:")
for label in results:
    print(f"  intermediary/clusters{label}.csv")

NR data: 6,480 rows
Countries: 54
Years: [np.float64(1995.0), np.float64(1996.0), np.float64(1997.0), np.float64(1998.0), np.float64(1999.0), np.float64(2000.0), np.float64(2001.0), np.float64(2002.0), np.float64(2003.0), np.float64(2004.0), np.float64(2005.0), np.float64(2006.0), np.float64(2007.0), np.float64(2008.0), np.float64(2009.0), np.float64(2010.0), np.float64(2011.0), np.float64(2012.0), np.float64(2013.0), np.float64(2014.0), np.float64(2015.0), np.float64(2016.0), np.float64(2017.0), np.float64(2018.0), np.float64(2019.0), np.float64(2020.0), np.float64(2021.0)]
Resources: 21

Silhouette scores:
  k=2: 0.483
  k=3: 0.509
  k=4: 0.529 <-- selected
  k=5: 0.553
  k=6: 0.557
  k=7: 0.536
  k=8: 0.544
1995 SNAPSHOT
Silhouette score: 0.529
Cluster distribution:
  Diversified Exporters: 7 countries
  Gold & Coal: 20 countries
  Oil Exporters: 13 countries
  Petrostates: 11 countries

2019 SNAPSHOT
Silhouette score: 0.465
Cluster distribution:
  Diversified Exporters: 7 countries

## Section 2 — ML & Regression Charts

In [1]:
"""
viz_updates.py — Apply all chart modifications from user feedback.
"""

def _hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


# =============================================================================
# CHART 09 — Train vs Test R² (XGBoost removed)
# =============================================================================
print("\n[09] Train vs Test R²...")

perf_l = pd.read_csv(os.path.join(NB5, 'model_performance_level.csv'))
perf_d = pd.read_csv(os.path.join(NB5, 'model_performance_delta.csv'))
perf_l = perf_l[perf_l['Model'] != 'XGBoost'].reset_index(drop=True)
perf_d = perf_d[perf_d['Model'] != 'XGBoost'].reset_index(drop=True)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14)

for col_idx, (perf, panel) in enumerate([(perf_l, 'ECI Level'), (perf_d, 'ΔECI')], 1):
    models   = perf['Model'].tolist()
    train_r2 = perf['Train R²'].tolist()
    test_r2  = perf['Test R²'].tolist()

    fig.add_trace(go.Bar(
        x=models, y=train_r2,
        marker=dict(color=PALETTE['blue'], line=dict(color='white', width=1)),
        name='Train R²', showlegend=(col_idx == 1), legendgroup='train',
        hovertemplate='%{x} Train: %{y:.3f}<extra></extra>',
        offsetgroup='train',
        text=[f'{v:.3f}' for v in train_r2],
        textposition='outside',
        textfont=dict(size=9, color=PALETTE['blue']),
    ), row=1, col=col_idx)

    fig.add_trace(go.Bar(
        x=models, y=test_r2,
        marker=dict(color=PALETTE['red'], line=dict(color='white', width=1)),
        name='Test R²', showlegend=(col_idx == 1), legendgroup='test',
        hovertemplate='%{x} Test: %{y:.3f}<extra></extra>',
        offsetgroup='test',
        text=[f'{v:.3f}' for v in test_r2],
        textposition='outside',
        textfont=dict(size=9, color=PALETTE['red']),
    ), row=1, col=col_idx)

    fig.update_yaxes(title_text='R²', gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)
    fig.update_xaxes(tickfont=dict(size=10), tickangle=-30, row=1, col=col_idx)

for x_paper, label in [(0.23, 'ECI Level'), (0.77, 'ΔECI')]:
    fig.add_annotation(
        x=x_paper, y=1.04, xref='paper', yref='paper',
        text=f'<b>{label}</b>', showarrow=False,
        font=dict(size=12, color=NAVY, family=FONT),
        xanchor='center', yanchor='bottom',
    )

fig.update_layout(
    barmode='group',
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    **base_layout(
        height=480, margin=dict(l=60, r=60, t=80, b=100),
        legend=dict(orientation='h', yanchor='bottom', y=1.06,
                    xanchor='center', x=0.5, font=dict(size=11)),
    ),
)
save(fig, '09_ml__train_vs_test_r2_all_models', OUT)


# =============================================================================
# CHART 10 — Actual vs Predicted (ECI level + ΔECI side by side)
# =============================================================================
print("\n[10] Actual vs Predicted — ECI + ΔECI...")

preds = pd.read_csv(os.path.join(NB5, 'test_predictions.csv'))

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
)

for col_idx, (actual_col, pred_col, label) in enumerate([
    ('Actual_ECI',   'Predicted_ECI',   'ECI'),
    ('Actual_Delta', 'Predicted_Delta', 'ΔECI'),
], 1):
    actual = preds[actual_col].dropna().values
    pred   = preds.loc[preds[actual_col].notna(), pred_col].values
    codes  = preds.loc[preds[actual_col].notna(), 'Country Code'].values
    names  = preds.loc[preds[actual_col].notna(), 'Country Name'].values

    lims = [min(actual.min(), pred.min()) - 0.1, max(actual.max(), pred.max()) + 0.1]
    mid  = 0.0

    for x0, x1, y0, y1, fc in [
        (lims[0], mid,     lims[0], mid,     'rgba(46,125,74,0.07)'),
        (mid,     lims[1], mid,     lims[1], 'rgba(46,125,74,0.07)'),
        (lims[0], mid,     mid,     lims[1], 'rgba(194,58,58,0.07)'),
        (mid,     lims[1], lims[0], mid,     'rgba(194,58,58,0.07)'),
    ]:
        fig.add_shape(type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
                      fillcolor=fc, line=dict(width=0), layer='below',
                      row=1, col=col_idx)

    fig.add_trace(go.Scatter(
        x=[lims[0], lims[1]], y=[lims[0], lims[1]],
        mode='lines', line=dict(color=PALETTE['red'], width=1.5, dash='dash'),
        name='45° line', showlegend=(col_idx == 1), legendgroup='line45',
    ), row=1, col=col_idx)

    resid   = np.abs(actual - pred)
    top_idx = set(np.argsort(resid)[::-1][:5])
    mask_n  = np.array([i not in top_idx for i in range(len(actual))])

    fig.add_trace(go.Scatter(
        x=actual[mask_n], y=pred[mask_n], mode='markers',
        marker=dict(size=6, color=PALETTE['blue'], opacity=0.65,
                    line=dict(color='white', width=0.5)),
        name='Test obs.', showlegend=(col_idx == 1), legendgroup='obs',
        customdata=np.stack([codes[mask_n], names[mask_n]], axis=1),
        hovertemplate='<b>%{customdata[1]}</b><br>'
                      'Actual: %{x:.3f}<br>Predicted: %{y:.3f}<extra></extra>',
    ), row=1, col=col_idx)

    out_idx = list(top_idx)
    fig.add_trace(go.Scatter(
        x=actual[out_idx], y=pred[out_idx], mode='markers+text',
        marker=dict(size=9, color=PALETTE['orange'], opacity=0.9,
                    line=dict(color='white', width=1)),
        text=codes[out_idx], textposition='top center', textfont=dict(size=9),
        name='Largest residuals', showlegend=(col_idx == 1), legendgroup='outliers',
        customdata=np.stack([codes[out_idx], names[out_idx]], axis=1),
        hovertemplate='<b>%{customdata[1]}</b><br>'
                      'Actual: %{x:.3f}<br>Predicted: %{y:.3f}<extra></extra>',
    ), row=1, col=col_idx)

    fig.add_hline(y=0, line=dict(color=GRID, width=1), row=1, col=col_idx)
    fig.add_vline(x=0, line=dict(color=GRID, width=1), row=1, col=col_idx)

    fig.update_xaxes(title_text=f'Actual {label} (test set)', range=lims,
                     gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)
    fig.update_yaxes(title_text=f'Predicted {label}', range=lims,
                     gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

fig.update_layout(**base_layout(
    height=560, margin=dict(l=70, r=50, t=70, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.04,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig, '10_ml__actual_vs_predicted_eci_test_set', OUT)


# =============================================================================
# CHART 11 — ECI Forecast: two panels (best performers | worst performers)
#            Each panel shows all 54 grey + case studies + top3 or bottom3
# =============================================================================
print("\n[11] ECI Forecast — split into best / worst panels...")

fc   = pd.read_csv(os.path.join(NB5, 'ECI_Forecast_2020_2030.csv'))
rank = pd.read_csv(os.path.join(NB5, 'Country_Ranking_2020_2030.csv'))
perf = pd.read_csv(os.path.join(NB5, 'model_performance_level.csv'))
perf = perf[perf['Model'] != 'XGBoost']
best_rmse = perf.iloc[0]['Test RMSE'] if 'Test RMSE' in perf.columns else 0.08

master = load_master()
hist   = master[['Country Code', 'Country Name', 'Year',
                  'Economic Complexity Index']].dropna()

rank_sorted  = rank.sort_values('Total_Change', ascending=False).reset_index(drop=True)
CASE_STUDIES = ['COG', 'AZE', 'CHL']
top3    = [cc for cc in rank_sorted['Country Code'].tolist() if cc not in CASE_STUDIES][:3]
bottom3 = [cc for cc in rank_sorted['Country Code'].tolist()[::-1] if cc not in CASE_STUDIES][:3]

CASE_COL = '#4a6fa5'
TOP_COL  = '#2e7d4a'
BOT_COL  = '#c23a3a'
GREY     = '#b0b8c4'

all_eci = pd.concat([
    hist[hist['Country Code'].isin(INCLUDE_LIST)]['Economic Complexity Index'],
    fc[fc['Country Code'].isin(INCLUDE_LIST)]['Ensemble'],
]).dropna()
Y_RANGE = [all_eci.min() - 0.15, all_eci.max() + 0.15]

def _add_country_traces(fig, cc, cname, col, lw, opacity, legendgroup,
                        row_n, col_n, show_legend=False):
    """Draw historical solid + forecast dashed (+ confidence band for highlights)."""
    h = hist[hist['Country Code'] == cc].sort_values('Year')
    f = fc[fc['Country Code'] == cc].sort_values('Year')
    if h.empty or f.empty:
        return None
    ens = f['Ensemble'].values
    yrs = f['Year'].values

    fig.add_trace(go.Scatter(
        x=h['Year'], y=h['Economic Complexity Index'],
        mode='lines', line=dict(color=col, width=lw), opacity=opacity,
        legendgroup=legendgroup, showlegend=False,
        hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Historical</extra>',
    ), row=row_n, col=col_n)

    if col != GREY:
        rgb = ','.join(str(v) for v in _hex_to_rgb(col))
        fig.add_trace(go.Scatter(
            x=np.concatenate([yrs, yrs[::-1]]).tolist(),
            y=np.concatenate([ens + best_rmse, (ens - best_rmse)[::-1]]).tolist(),
            fill='toself', fillcolor=f'rgba({rgb},0.08)',
            line=dict(color='rgba(0,0,0,0)'),
            showlegend=False, hoverinfo='skip', legendgroup=legendgroup,
        ), row=row_n, col=col_n)

    last_yr  = int(h['Year'].iloc[-1])
    last_eci = float(h['Economic Complexity Index'].iloc[-1])
    fig.add_trace(go.Scatter(
        x=[last_yr] + yrs.tolist(), y=[last_eci] + ens.tolist(),
        mode='lines', line=dict(color=col, width=lw, dash='dash'), opacity=opacity,
        legendgroup=legendgroup, showlegend=False,
        hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Forecast</extra>',
    ), row=row_n, col=col_n)
    return float(ens[-1])


def _deconflict(label_info, min_gap=0.22):
    sorted_lbl = sorted(label_info.items(), key=lambda x: x[1][0])
    adjusted   = {}
    floor_y    = None
    for cc, (y_act, _) in sorted_lbl:
        y_place = y_act if floor_y is None else max(y_act, floor_y + min_gap)
        adjusted[cc] = y_place
        floor_y = y_place
    return adjusted


def _add_labels(fig, label_info, adjusted_y, x_anchor=2031):
    for cc, (y_act, col) in label_info.items():
        fig.add_annotation(
            x=2030, y=y_act,
            ax=x_anchor, ay=adjusted_y[cc],
            axref='x', ayref='y',
            text=f'<b>{cc}</b>',
            showarrow=True,
            arrowhead=2, arrowwidth=1, arrowsize=0.8, arrowcolor=col,
            font=dict(size=9.5, color=col, family=FONT),
            xanchor='left', yanchor='middle',
        )


fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.07,
)

for panel_col, highlight_group, highlight_col, grp_name in [
    (1, top3,    TOP_COL, 'top3'),
    (2, bottom3, BOT_COL, 'bottom3'),
]:
    # shared x-axis decorations
    fig.add_vrect(x0=2019.5, x1=2030.5, fillcolor='rgba(200,210,225,0.18)',
                  line=dict(width=0), layer='below', row=1, col=panel_col)
    fig.add_vline(x=2019.5, line=dict(color='#aaa', width=1.5, dash='dot'),
                  row=1, col=panel_col)

    highlighted_here = set(CASE_STUDIES + highlight_group)

    # 1. Grey background — all non-highlighted
    for cc in INCLUDE_LIST:
        if cc in highlighted_here:
            continue
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        _add_country_traces(fig, cc, cname_, GREY, lw=0.7, opacity=0.3,
                            legendgroup='others', row_n=1, col_n=panel_col)

    label_info = {}

    # 2. Highlight group (top3 or bottom3)
    for cc in highlight_group:
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        y_end  = _add_country_traces(fig, cc, cname_, highlight_col, lw=2.2, opacity=1.0,
                                     legendgroup=grp_name, row_n=1, col_n=panel_col)
        if y_end is not None:
            label_info[cc] = (y_end, highlight_col)

    # 3. Case studies
    for cc in CASE_STUDIES:
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        y_end  = _add_country_traces(fig, cc, cname_, CASE_COL, lw=2.5, opacity=1.0,
                                     legendgroup='cases', row_n=1, col_n=panel_col)
        if y_end is not None:
            label_info[cc] = (y_end, CASE_COL)

    adjusted_y = _deconflict(label_info)

    # Annotations use panel-specific xref/yref
    xref = 'x' if panel_col == 1 else 'x2'
    yref = 'y' if panel_col == 1 else 'y2'
    for cc, (y_act, col) in label_info.items():
        fig.add_annotation(
            x=2030, y=y_act,
            ax=2031, ay=adjusted_y[cc],
            axref=xref, ayref=yref,
            xref=xref, yref=yref,
            text=f'<b>{cc}</b>',
            showarrow=True,
            arrowhead=2, arrowwidth=1, arrowsize=0.8, arrowcolor=col,
            font=dict(size=9.5, color=col, family=FONT),
            xanchor='left', yanchor='middle',
        )

    fig.update_xaxes(title_text='Year', gridcolor=GRID, gridwidth=0.5,
                     dtick=5, range=[1994, 2033], row=1, col=panel_col)
    fig.update_yaxes(title_text='Economic Complexity Index' if panel_col == 1 else '',
                     range=Y_RANGE, gridcolor=GRID, gridwidth=0.5, row=1, col=panel_col)

# Shared legend entries
for lbl, col, rk, grp in [
    ('Case studies — COG · AZE · CHL',       CASE_COL, 1, 'cases'),
    ('Top 3 improvers — GNQ · MNG · ECU',    TOP_COL,  2, 'top3'),
    ('Bottom 3 decliners — ZWE · SAU · KAZ', BOT_COL,  3, 'bottom3'),
    ('Other countries',                       GREY,     4, 'others'),
]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col, width=2.5), name=lbl,
        legendgroup=grp, showlegend=True, legendrank=rk))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(color='#888', width=1.5), name='── Historical',
    showlegend=True, legendrank=10))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(color='#888', width=1.5, dash='dash'), name='- - Forecast',
    showlegend=True, legendrank=11))

fig.update_layout(**base_layout(
    height=620, margin=dict(l=70, r=20, t=70, b=110),
    legend=dict(
        orientation='h', font=dict(size=9.5),
        bgcolor='rgba(255,255,255,0.92)', bordercolor=GRID, borderwidth=1,
        x=0.0, y=-0.15, xanchor='left', yanchor='top', tracegroupgap=0,
    ),
))
save(fig, '11_ml__eci_forecast_top_improvers_2020_2030', OUT)
print("\n[11] ECI Forecast — split into best / worst panels...")

fc   = pd.read_csv(os.path.join(NB5, 'ECI_Forecast_2020_2030.csv'))
rank = pd.read_csv(os.path.join(NB5, 'Country_Ranking_2020_2030.csv'))
perf = pd.read_csv(os.path.join(NB5, 'model_performance_level.csv'))
perf = perf[perf['Model'] != 'XGBoost']
best_rmse = perf.iloc[0]['Test RMSE'] if 'Test RMSE' in perf.columns else 0.08

master = load_master()
hist   = master[['Country Code', 'Country Name', 'Year',
                  'Economic Complexity Index']].dropna()

rank_sorted  = rank.sort_values('Total_Change', ascending=False).reset_index(drop=True)
top3    = rank_sorted['Country Code'].tolist()[:3]
bottom3 = rank_sorted['Country Code'].tolist()[-3:][::-1]

TOP_COL  = '#2e7d4a'
BOT_COL  = '#c23a3a'
GREY     = '#b0b8c4'

all_eci = pd.concat([
    hist[hist['Country Code'].isin(INCLUDE_LIST)]['Economic Complexity Index'],
    fc[fc['Country Code'].isin(INCLUDE_LIST)]['Ensemble'],
]).dropna()
Y_RANGE = [all_eci.min() - 0.15, all_eci.max() + 0.15]

def _add_country_traces(fig, cc, cname, col, lw, opacity, legendgroup,
                        row_n, col_n, show_legend=False):
    """Draw historical solid + forecast dashed (+ confidence band for highlights)."""
    h = hist[hist['Country Code'] == cc].sort_values('Year')
    f = fc[fc['Country Code'] == cc].sort_values('Year')
    if h.empty or f.empty:
        return None
    ens = f['Ensemble'].values
    yrs = f['Year'].values

    fig.add_trace(go.Scatter(
        x=h['Year'], y=h['Economic Complexity Index'],
        mode='lines', line=dict(color=col, width=lw), opacity=opacity,
        legendgroup=legendgroup, showlegend=False,
        hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Historical</extra>',
    ), row=row_n, col=col_n)

    if col != GREY:
        rgb = ','.join(str(v) for v in _hex_to_rgb(col))
        fig.add_trace(go.Scatter(
            x=np.concatenate([yrs, yrs[::-1]]).tolist(),
            y=np.concatenate([ens + best_rmse, (ens - best_rmse)[::-1]]).tolist(),
            fill='toself', fillcolor=f'rgba({rgb},0.08)',
            line=dict(color='rgba(0,0,0,0)'),
            showlegend=False, hoverinfo='skip', legendgroup=legendgroup,
        ), row=row_n, col=col_n)

    last_yr  = int(h['Year'].iloc[-1])
    last_eci = float(h['Economic Complexity Index'].iloc[-1])
    fig.add_trace(go.Scatter(
        x=[last_yr] + yrs.tolist(), y=[last_eci] + ens.tolist(),
        mode='lines', line=dict(color=col, width=lw, dash='dash'), opacity=opacity,
        legendgroup=legendgroup, showlegend=False,
        hovertemplate=f'<b>{cname} ({cc})</b><br>%{{x}}: %{{y:.3f}}<extra>Forecast</extra>',
    ), row=row_n, col=col_n)
    return float(ens[-1])


def _deconflict(label_info, min_gap=0.22):
    sorted_lbl = sorted(label_info.items(), key=lambda x: x[1][0])
    adjusted   = {}
    floor_y    = None
    for cc, (y_act, _) in sorted_lbl:
        y_place = y_act if floor_y is None else max(y_act, floor_y + min_gap)
        adjusted[cc] = y_place
        floor_y = y_place
    return adjusted


fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.07,
)

for panel_col, highlight_group, highlight_col, grp_name in [
    (1, top3,    TOP_COL, 'top3'),
    (2, bottom3, BOT_COL, 'bottom3'),
]:
    fig.add_vrect(x0=2019.5, x1=2030.5, fillcolor='rgba(200,210,225,0.18)',
                  line=dict(width=0), layer='below', row=1, col=panel_col)
    fig.add_vline(x=2019.5, line=dict(color='#aaa', width=1.5, dash='dot'),
                  row=1, col=panel_col)

    highlighted_here = set(highlight_group)

    # 1. Grey background — all non-highlighted
    for cc in INCLUDE_LIST:
        if cc in highlighted_here:
            continue
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        _add_country_traces(fig, cc, cname_, GREY, lw=0.7, opacity=0.3,
                            legendgroup='others', row_n=1, col_n=panel_col)

    label_info = {}

    # 2. Highlight group (top3 or bottom3)
    for cc in highlight_group:
        row_   = rank[rank['Country Code'] == cc]
        cname_ = row_['Country'].values[0] if len(row_) else cc
        y_end  = _add_country_traces(fig, cc, cname_, highlight_col, lw=2.2, opacity=1.0,
                                     legendgroup=grp_name, row_n=1, col_n=panel_col)
        if y_end is not None:
            label_info[cc] = (y_end, highlight_col)

    adjusted_y = _deconflict(label_info)

    xref = 'x' if panel_col == 1 else 'x2'
    yref = 'y' if panel_col == 1 else 'y2'
    for cc, (y_act, col) in label_info.items():
        fig.add_annotation(
            x=2030, y=y_act,
            ax=2031, ay=adjusted_y[cc],
            axref=xref, ayref=yref,
            xref=xref, yref=yref,
            text=f'<b>{cc}</b>',
            showarrow=True,
            arrowhead=2, arrowwidth=1, arrowsize=0.8, arrowcolor=col,
            font=dict(size=9.5, color=col, family=FONT),
            xanchor='left', yanchor='middle',
        )

    fig.update_xaxes(title_text='Year', gridcolor=GRID, gridwidth=0.5,
                     dtick=5, range=[1994, 2033], row=1, col=panel_col)
    fig.update_yaxes(title_text='Economic Complexity Index' if panel_col == 1 else '',
                     range=Y_RANGE, gridcolor=GRID, gridwidth=0.5, row=1, col=panel_col)

# Shared legend entries (no case studies)
for lbl, col, rk, grp in [
    ('Top 3 improvers',    TOP_COL,  1, 'top3'),
    ('Bottom 3 decliners', BOT_COL,  2, 'bottom3'),
    ('Other countries',    GREY,     3, 'others'),
]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col, width=2.5), name=lbl,
        legendgroup=grp, showlegend=True, legendrank=rk))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(color='#888', width=1.5), name='── Historical',
    showlegend=True, legendrank=10))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
    line=dict(color='#888', width=1.5, dash='dash'), name='- - Forecast',
    showlegend=True, legendrank=11))

fig.update_layout(**base_layout(
    height=620, margin=dict(l=70, r=20, t=70, b=120),
    legend=dict(
        orientation='h', font=dict(size=15),
        bgcolor='rgba(255,255,255,0.92)', bordercolor=GRID, borderwidth=1,
        x=0.5, y=-0.15, xanchor='center', yanchor='top', tracegroupgap=5,
    ),
))
save(fig, '11b_ml__eci_forecast_top_improvers_2020_2030', OUT)

# =============================================================================
# CHART 15 — ECI Cluster Trajectories (cluster names instead of numbers)
# =============================================================================
print("\n[15] ECI Cluster Trajectories — cluster names...")

master   = load_master()
clusters = load_clusters('1995')[['Country Code', 'Cluster']].drop_duplicates()
df       = master[master['Country Code'].isin(INCLUDE_LIST)].copy()
df       = df.merge(clusters, on='Country Code', how='left')

traj = (df.groupby(['Year', 'Cluster'])['Economic Complexity Index']
          .median().reset_index())

fig = go.Figure()
for cl in sorted(traj['Cluster'].dropna().unique()):
    sub = traj[traj['Cluster'] == cl]
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers',
        name=CLUSTER_LABELS.get(int(cl), f'Cluster {int(cl)}'),
        line=dict(color=CLUSTER_COLORS.get(int(cl), '#999'), width=2.2),
        marker=dict(size=5),
        hovertemplate='%{x}: %{y:.3f}<extra>' +
                      CLUSTER_LABELS.get(int(cl), '') + '</extra>',
    ))

fig.update_layout(**base_layout(
    height=480,
    xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5, dtick=5),
    yaxis=dict(title='Median ECI', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
    hovermode='x unified',
))
save(fig, '15_reg__eci_mean_trajectory_by_cluster', OUT)


# =============================================================================
# CHART 16 — Coefficients 3a vs 3b (remove Driscoll-Kraay from axis title)
# =============================================================================
print("\n[16] Coefficients 3a vs 3b — removing Driscoll-Kraay...")

chart16_path = os.path.join(OUT, '16_reg__coefficients_model3a_vs_model3b.html')
if os.path.exists(chart16_path):
    html = open(chart16_path, encoding='utf-8').read()
    html = html.replace('95 % CI, Driscoll-Kraay', '95% CI')
    html = html.replace('95% CI, Driscoll-Kraay', '95% CI')
    html = html.replace('Driscoll-Kraay', '')
    open(chart16_path, 'w', encoding='utf-8').write(html)
    print(f"  Updated: 16_reg__coefficients_model3a_vs_model3b.html")


# =============================================================================
# CHART 17 — HCI × Production interaction (simplified: country means)
# =============================================================================
print("\n[17] HCI × Production interaction — simplifying...")

master = load_master()
df     = master[master['Country Code'].isin(INCLUDE_LIST)].copy()
df['prod_pc']    = df['Total_Production_Value'] / df['Population'].replace(0, np.nan)
df['log_HCI']    = np.log1p(df['Human capital index'])
df['log_prod_pc']= np.log1p(df['prod_pc'])

# Aggregate to country means — one point per country, much cleaner than all obs
country_avg = (
    df[['Country Code', 'log_HCI', 'Economic Complexity Index', 'log_prod_pc']]
    .dropna()
    .groupby('Country Code')
    .mean()
    .reset_index()
)

# Assign production quartile based on country average
country_avg['Prod_quartile'] = pd.qcut(
    country_avg['log_prod_pc'], q=4,
    labels=['Q1 — Low production', 'Q2', 'Q3', 'Q4 — High production']
)

q_colors = [PALETTE['light_blue'], PALETTE['blue'], PALETTE['orange'], PALETTE['red']]

fig = go.Figure()
for q, col in zip(['Q1 — Low production', 'Q2', 'Q3', 'Q4 — High production'], q_colors):
    sub = country_avg[country_avg['Prod_quartile'] == q]
    fig.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub['Economic Complexity Index'],
        mode='markers+text',
        text=sub['Country Code'],
        textposition='top center',
        textfont=dict(size=8, color='#555'),
        marker=dict(color=col, size=9, opacity=0.85,
                    line=dict(width=0.8, color='white')),
        name=f'Production {q}',
        hovertemplate='<b>%{text}</b><br>log(HCI): %{x:.2f}<br>ECI: %{y:.2f}<extra></extra>',
    ))

fig.update_layout(**base_layout(
    height=540,
    xaxis=dict(title='log(Human Capital Index)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5),
    legend=dict(title=dict(text='Avg. Production p.c. Quartile'),
                font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
))
save(fig, '17_reg__hci_production_interaction_effect_on_eci', OUT)


# =============================================================================
# CHART 26 — PCA Resource Loadings Heatmap
#   - Red-white-blue colorscale (was yellow-white-blue)
#   - PC1 row at top
#   - Larger colorbar and chart
# =============================================================================
print("\n[26] PCA Loadings heatmap — fixing colours, PC1 first, larger...")

nr        = load_nr()
nr_sample = nr[nr['Country Code'].isin(INCLUDE_LIST)]
nr_1995   = nr_sample[nr_sample['Year'] == 1995]

pivot = nr_1995.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Population'],
    columns='Resource', values='Production_TotalValue',
).reset_index()

resource_cols = [c for c in pivot.columns
                 if c not in ['Country', 'Country Code', 'Year', 'Population']]
pivot[resource_cols] = pivot[resource_cols].div(pivot['Population'], axis=0)
pivot = pivot.fillna(0)

X   = np.log1p(pivot[resource_cols].fillna(0))
pca = PCA(n_components=2, random_state=42)
pca.fit(X)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100

loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=resource_cols)
top20    = loadings.abs().sum(axis=1).nlargest(20).index
plot_df  = loadings.loc[top20]
plot_df  = (plot_df
            .assign(_s=plot_df['PC1'].abs() + plot_df['PC2'].abs())
            .sort_values('_s', ascending=False)
            .drop(columns='_s'))

# PC labels — PC1 first = top row (reversed y in Plotly)
# PC1 loads on oil & gas (hydrocarbons); PC2 loads on copper, gold & coal
pc_labels_ordered = [
    f'PC1 ({var1:.1f}%)<br><i>↑ Oil & Gas</i>',
    f'PC2 ({var2:.1f}%)<br><i>↑ Copper, Gold & Coal</i>',
]
# Plotly heatmap y goes bottom→top, so put PC2 first in the list → PC1 on top
y_labels = pc_labels_ordered[::-1]          # [PC2, PC1] → displayed bottom→top = PC2 bottom, PC1 top
z_values = plot_df[['PC2', 'PC1']].T.values  # rows match y_labels order

fig = go.Figure(go.Heatmap(
    z=z_values,
    x=plot_df.index.tolist(),
    y=y_labels,
    colorscale=[
        [0.0, '#c23a3a'],
        [0.5, '#ffffff'],
        [1.0, '#1a4a8a'],
    ],
    zmid=0, zmin=-1, zmax=1,
    text=z_values.round(2),
    texttemplate='%{text:.2f}',
    textfont=dict(size=8, family=FONT),
    hovertemplate='%{x} / %{y}: %{z:.3f}<extra></extra>',
    colorbar=dict(
        title=dict(text='Loading', font=dict(size=12)),
        thickness=20, len=1.0,
        tickvals=[-1, -0.5, 0, 0.5, 1],
        tickfont=dict(size=11),
    ),
))

fig.update_xaxes(tickangle=-40, tickfont=dict(size=10, family=FONT), showgrid=False)
fig.update_yaxes(tickfont=dict(size=12, family=FONT), showgrid=False)
fig.update_layout(**base_layout(
    height=420,
    margin=dict(l=260, r=120, t=60, b=160),
))
save(fig, '26_diag__pca_resource_loadings_heatmap', OUT, w=1300, h=420)


# =============================================================================
# CHART 31 — ML Prediction Intervals (aggregated by country)
# =============================================================================
print("\n[31] Prediction intervals — aggregating by country...")

preds = pd.read_csv(os.path.join(NB5, 'test_predictions.csv'))

# Country-level: mean actual, mean predicted, std actual over test years
country_stats = (
    preds.groupby(['Country Code', 'Country Name'])
    .agg(
        Actual_mean   = ('Actual_ECI', 'mean'),
        Predicted_mean= ('Predicted_ECI', 'mean'),
        Actual_std    = ('Actual_ECI', 'std'),
        n_years       = ('Year', 'count'),
    )
    .reset_index()
    .sort_values('Actual_mean')
    .reset_index(drop=True)
)
country_stats['Actual_std'] = country_stats['Actual_std'].fillna(0)

# In-band: |actual_mean - predicted_mean| < 1 std dev
country_stats['In_band'] = (
    (country_stats['Actual_mean'] - country_stats['Predicted_mean']).abs()
    < country_stats['Actual_std']
)

fig = go.Figure()

# Std-dev band per country (horizontal error bar on actual)
fig.add_trace(go.Scatter(
    x=list(range(len(country_stats))) * 2 + list(range(len(country_stats)))[::-1] * 2,
    y=(country_stats['Actual_mean'] + country_stats['Actual_std']).tolist() +
      (country_stats['Actual_mean'] - country_stats['Actual_std']).iloc[::-1].tolist(),
    fill='toself', fillcolor='rgba(74,111,165,0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    hoverinfo='skip', name='±1 SD (actual ECI in test years)',
))

fig.add_trace(go.Scatter(
    x=list(range(len(country_stats))),
    y=country_stats['Actual_mean'],
    mode='lines', line=dict(color=PALETTE['blue'], width=2),
    name='Mean Actual ECI',
))

for in_band, color, sym, lbl in [
    (True,  PALETTE['green'], 'circle',  'Predicted ≈ Actual (within ±1 SD)'),
    (False, PALETTE['red'],   'diamond', 'Predicted outside ±1 SD'),
]:
    mask = country_stats['In_band'] == in_band
    sub  = country_stats[mask]
    fig.add_trace(go.Scatter(
        x=sub.index.tolist(),
        y=sub['Predicted_mean'],
        mode='markers',
        marker=dict(color=color, size=8 if in_band else 10, symbol=sym, opacity=0.85),
        name=lbl,
        customdata=sub[['Country Code', 'Country Name']].values,
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]})<br>'
                      'Avg Actual: %{text}<br>Avg Predicted: %{y:.3f}<extra></extra>',
        text=[f'{v:.3f}' for v in sub['Actual_mean']],
    ))

fig.update_layout(**base_layout(
    height=500,
    xaxis=dict(
        title='Countries (sorted by mean actual ECI)',
        tickvals=list(range(len(country_stats))),
        ticktext=country_stats['Country Code'].tolist(),
        tickangle=-60, tickfont=dict(size=8),
        gridcolor=GRID,
    ),
    yaxis=dict(title='ECI (test set mean)', gridcolor=GRID, gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig, '31_diag__ml_prediction_intervals', OUT)


# =============================================================================
# CHART 32 — Country Data Coverage (highlight high-missingness countries)
# =============================================================================
print("\n[32] Country data coverage — highlighting high-missingness countries...")

raw = load_master_wide()
sample_df = raw[raw['Country Code'].isin(INCLUDE_LIST)].copy()
country_missing = analyze_country_missingness(sample_df)

LABEL_THRESHOLD = 20.0   # label countries above this % missing

fig = go.Figure()

for above in [True, False]:
    mask = country_missing['% Missing'] >= LABEL_THRESHOLD if above else \
           country_missing['% Missing'] < LABEL_THRESHOLD
    sub  = country_missing[mask]
    color= PALETTE['red'] if above else PALETTE['blue']
    size = 12 if above else 8
    mode = 'markers+text' if above else 'markers'

    fig.add_trace(go.Scatter(
        x=sub['Vars with Data'],
        y=sub['% Missing'],
        mode=mode,
        text=sub['Code'] if above else None,
        textposition='top center',
        textfont=dict(size=9, color=PALETTE['red']),
        marker=dict(color=color, size=size, opacity=0.75 if above else 0.55,
                    line=dict(color='white', width=0.8)),
        name=f'>= {LABEL_THRESHOLD}% missing' if above else f'< {LABEL_THRESHOLD}% missing',
        customdata=sub[['Code', 'Country', 'Complete Vars', 'Years Covered', 'Rows']].values,
        hovertemplate=(
            '<b>%{customdata[1]}</b> (%{customdata[0]})<br>'
            'Vars with data: %{x}<br>'
            '% Missing: %{y:.1f}%<br>'
            'Complete vars: %{customdata[2]}<br>'
            'Years covered: %{customdata[3]}<extra></extra>'
        ),
    ))

med_vars    = country_missing['Vars with Data'].median()
med_missing = country_missing['% Missing'].median()
fig.add_hline(y=med_missing, line_dash='dash', line_color='#aaa', opacity=0.6,
              annotation_text=f'Median {med_missing:.1f}%', annotation_position='right')
fig.add_vline(x=med_vars, line_dash='dash', line_color='#aaa', opacity=0.6,
              annotation_text=f'Median {med_vars:.0f} vars', annotation_position='top')

fig.update_layout(**base_layout(
    height=520,
    xaxis=dict(title='Variables with Any Data', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='% Missing Data Overall', gridcolor=GRID, gridwidth=0.5),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
))
save(fig, '32_diag__country_data_coverage_scatter', OUT)


print("\nAll updates complete.")


[09] Train vs Test R²...
  Saved: Final/charts\09_ml__train_vs_test_r2_all_models.html

[10] Actual vs Predicted — ECI + ΔECI...
  Saved: Final/charts\10_ml__actual_vs_predicted_eci_test_set.html

[11] ECI Forecast — split into best / worst panels...
  Saved: Final/charts\11_ml__eci_forecast_top_improvers_2020_2030.html

[11] ECI Forecast — split into best / worst panels...
  Saved: Final/charts\11b_ml__eci_forecast_top_improvers_2020_2030.html

[15] ECI Cluster Trajectories — cluster names...
  Saved: Final/charts\15_reg__eci_mean_trajectory_by_cluster.html

[16] Coefficients 3a vs 3b — removing Driscoll-Kraay...
  Updated: 16_reg__coefficients_model3a_vs_model3b.html

[17] HCI × Production interaction — simplifying...
  Saved: Final/charts\17_reg__hci_production_interaction_effect_on_eci.html

[26] PCA Loadings heatmap — fixing colours, PC1 first, larger...
  Saved: Final/charts\26_diag__pca_resource_loadings_heatmap.html

[31] Prediction intervals — aggregating by country...
  Save